In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/age_size/validation/ibm'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# Offline Age–Size ABC–IPM Point Estimates

This notebook compares Exact-GP and DeepRV simulation using the **same 5,000 modular parameter combinations**, the **same observed summaries**, and **common Exact-GP thresholds**. It uses the 50 saved simulations per parameter combination and method; it does not run new IBMs, call the Transformer, or rerun full ABC-PMC.

For parameter combination $i$ and replicate $r$, the selected year–summary discrepancies are combined as

$$D_{ir}=\sqrt{\sum_{j\in\mathcal I}\left(s_{irj}/\operatorname{MAD}_j\right)^2},$$

where $\mathcal I$ contains components with a positive Exact-GP MAD. Three Year 1 reproduction components are structurally zero and are excluded, leaving 47 informative components. The MADs and thresholds are calculated from Exact-GP simulations only. At a common threshold $\epsilon$, the weight of parameter combination $i$ is proportional to its estimated acceptance probability, $\hat p_i=R^{-1}\sum_r I(D_{ir}\leq\epsilon)$.

One age–size IPM is built for each parameter combination. Exact-GP and DeepRV weights are then applied to the same IPM scores. Results are descriptive point estimates; there is no bootstrap or uncertainty interval.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import gc
import hashlib
import sys

import gpflow
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigs
from scipy.stats import norm

def find_project_directory(start):
    for directory in (start, *start.parents):
        if (directory / "pyproject.toml").is_file():
            return directory
    raise FileNotFoundError("Could not find the dl4bi project directory")

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

project_directory = PROJECT_ROOT
sys.path.insert(0, str(project_directory))

result_directory = GENERATED / 'soay/age_size/validation/ibm'
design_file = result_directory / "benchmark_design.npz"
samples_file = GENERATED / 'soay/age_size/inference/gpdf_agesize_gp_samples.npz'
data_file = GENERATED / 'soay/age_size/data/gp_ibm_dataset_agesize.csv'
true_metrics_file = GENERATED / 'soay/age_size/data/true_gp_metrics.npz'
distance_cache_file = result_directory / "offline_scalar_distances.npz"
prediction_cache_file = result_directory / "offline_vital_rate_predictions.npz"
ipm_cache_file = result_directory / "offline_ipm_scores.npz"
point_estimate_file = result_directory / "offline_ipm_point_estimates.csv"
weight_file = result_directory / "offline_rejection_weights.npz"
diagnostic_file = result_directory / "offline_weight_diagnostics.csv"
difference_file = result_directory / "offline_method_differences.csv"

selected_summaries = [
    "emd_breeder_next_size_distribution",
    "chi2_total_survivors_by_size_age",
    "chi2_total_breeders_by_size",
    "chi2_total_recruits_by_age",
    "bhattacharyya_nonbreeder_next_size_distribution",
]
threshold_quantiles = np.array([0.50, 0.20, 0.10, 0.05, 0.01])
maximum_load_workers = 8
lower_size, upper_size, number_mesh_points = 2.0, 3.5, 250
maximum_natural_age = 7
minimum_reproductive_age = 1
female_probability = 0.5
rate_columns = {"growth": 0, "surv": 1, "repr": 2, "recr": 3, "rcsz": 4}
method_styles = {
    "Exact GP": {"color": "#0072B2", "linestyle": "-"},
    "DeepRV": {"color": "#D55E00", "linestyle": "--"},
}
figure_width = 5.906
plt.style.use("seaborn-v0_8-white")
gpflow.config.set_default_float(np.float64)
print(f"Project Directory: {project_directory}")

## Load And Validate The Fixed Design

Each row of `parameter_indices` is one modular combination containing posterior indices for growth, survival, reproduction, recruitment and recruit size. The same row was used for its 50 Exact-GP simulations and 50 DeepRV simulations.

In [ ]:
with np.load(design_file) as design:
    parameter_indices = design["parameter_indices"].copy()
    summary_names = design["summary_names"].astype(str)
    model_names = design["model_names"].astype(str)
    years = design["years"].copy()
    number_parameters = int(design["number_parameters"].item())
    number_replicates = int(design["number_replicates"].item())
    sample_hash = str(design["samples_sha256"].item())

design_hash = file_sha256(design_file)
if file_sha256(samples_file) != sample_hash:
    raise ValueError("The GP sample archive does not match the validation design")
if parameter_indices.shape != (number_parameters, len(rate_columns)):
    raise ValueError("Unexpected parameter-index shape")
if tuple(model_names) != ("m_grw", "m_surv", "m_repr", "m_recr", "m_rcsz"):
    raise ValueError("Unexpected vital-rate order in the validation design")
if not set(selected_summaries).issubset(summary_names):
    raise ValueError("A selected ABC summary is absent from the validation design")
summary_indices = np.array([np.flatnonzero(summary_names == name).item() for name in selected_summaries])

print(f"Parameter Combinations: {number_parameters:,}")
print(f"Replicates Per Combination And Method: {number_replicates}")
print(f"Years: {years.tolist()}")
print(f"Selected Summaries: {selected_summaries}")

## Construct Scalar ABC Distances

The first run opens the raw chunk files and writes a compact cache. Infinite discrepancies remain infinite, so a trajectory with an extinction or overflow cannot pass a finite threshold. Components with zero Exact-GP MAD are identified explicitly and excluded from the scalar distance.

In [ ]:
def read_selected_chunk(arguments):
    chunk_path, indices = arguments
    with np.load(chunk_path) as chunk:
        if np.any(chunk["error"] != ""):
            raise ValueError(f"Simulation error in {chunk_path}")
        return chunk["parameter_id"].astype(int), chunk["replicate_id"].astype(int), chunk["distances"][:, :, indices]

def load_selected_summaries(method_directory):
    shape = (number_parameters, number_replicates, len(years), len(summary_indices))
    values = np.full(shape, np.nan)
    seen = np.zeros(shape[:2], dtype=bool)
    chunk_paths = sorted(method_directory.glob("chunk_*.npz"))
    if not chunk_paths:
        raise FileNotFoundError(f"No chunks found in {method_directory}")
    arguments = ((path, summary_indices) for path in chunk_paths)
    with ThreadPoolExecutor(max_workers=maximum_load_workers) as pool:
        for chunk_number, (parameter_id, replicate_id, distances) in enumerate(pool.map(read_selected_chunk, arguments), start=1):
            if np.any(seen[parameter_id, replicate_id]):
                raise ValueError(f"Duplicate result in {chunk_paths[chunk_number - 1]}")
            values[parameter_id, replicate_id] = distances
            seen[parameter_id, replicate_id] = True
            if chunk_number % 250 == 0 or chunk_number == len(chunk_paths):
                print(f"{method_directory.name}: {chunk_number:,}/{len(chunk_paths):,} Chunks", flush=True)
    if not seen.all():
        raise ValueError(f"{method_directory} is missing {(~seen).sum():,} results")
    return values

def scalar_distances(values, component_mad, informative_components):
    selected_values = values[:, :, informative_components]
    selected_mad = component_mad[informative_components]
    return np.sqrt(np.sum((selected_values / selected_mad[None, None, :]) ** 2, axis=2))

if distance_cache_file.exists():
    with np.load(distance_cache_file) as cache:
        if str(cache["design_hash"].item()) != design_hash:
            raise ValueError("The scalar-distance cache uses a different validation design")
        if cache["selected_summaries"].astype(str).tolist() != selected_summaries:
            raise ValueError("The scalar-distance cache uses different summaries")
        exact_distances = cache["exact_distances"].copy()
        deeprv_distances = cache["deeprv_distances"].copy()
        component_mad = cache["component_mad"].copy()
        informative_components = cache["informative_components"].astype(bool)
    print(f"Loaded {distance_cache_file.name}")
else:
    exact_summaries = load_selected_summaries(result_directory / "exact_gp")
    finite_exact = np.where(np.isfinite(exact_summaries), exact_summaries, np.nan)
    component_median = np.nanmedian(finite_exact, axis=(0, 1))
    component_mad = np.nanmedian(np.abs(finite_exact - component_median[None, None, :, :]), axis=(0, 1))
    informative_components = np.isfinite(component_mad) & (component_mad > 0)
    if not informative_components.any():
        raise ValueError("No year-summary component has a positive Exact-GP MAD")
    exact_distances = scalar_distances(exact_summaries, component_mad, informative_components)
    del exact_summaries, finite_exact
    deeprv_summaries = load_selected_summaries(result_directory / "deeprv")
    deeprv_distances = scalar_distances(deeprv_summaries, component_mad, informative_components)
    del deeprv_summaries
    np.savez_compressed(distance_cache_file, exact_distances=exact_distances, deeprv_distances=deeprv_distances, component_mad=component_mad, informative_components=informative_components, selected_summaries=np.asarray(selected_summaries), design_hash=np.asarray(design_hash))
    print(f"Saved {distance_cache_file.name}")

excluded_components = {(int(years[year_index]), selected_summaries[statistic_index]) for year_index, statistic_index in np.argwhere(~informative_components)}
expected_excluded_components = {
    (1, "emd_breeder_next_size_distribution"),
    (1, "chi2_total_breeders_by_size"),
    (1, "chi2_total_recruits_by_age"),
}
if excluded_components != expected_excluded_components:
    raise ValueError(f"Unexpected zero-MAD components: {sorted(excluded_components)}")

expected_shape = (number_parameters, number_replicates)
if exact_distances.shape != expected_shape or deeprv_distances.shape != expected_shape:
    raise ValueError("Unexpected scalar-distance shape")

mad_rows = []
for year_index, year in enumerate(years):
    for statistic_index, statistic in enumerate(selected_summaries):
        mad_rows.append({"Year": int(year), "Statistic": statistic, "Exact GP MAD": component_mad[year_index, statistic_index], "Included": bool(informative_components[year_index, statistic_index])})
mad_table = pd.DataFrame(mad_rows)
print(f"Informative Components: {informative_components.sum()} Of {informative_components.size}")
display(mad_table.loc[~mad_table["Included"]])
display(mad_table.pivot(index="Statistic", columns="Year", values="Exact GP MAD").round(4))

## Common Thresholds And Rejection Weights

Each threshold is a quantile of all $5{,}000\times50$ Exact-GP scalar distances, including infinite distances from unusable trajectories. The same numerical threshold is applied to both methods. Because the 5,000 design points have equal prior mass, their normalized acceptance frequencies are the rejection-ABC weights.

In [ ]:
if np.isnan(exact_distances).any() or np.isnan(deeprv_distances).any():
    raise ValueError("NaN scalar distances were found")
if np.mean(np.isfinite(exact_distances)) <= threshold_quantiles.max():
    raise ValueError("Too few finite Exact-GP distances for the requested thresholds")

thresholds = np.quantile(exact_distances.ravel(), threshold_quantiles)
weights = {}
diagnostic_rows = []
for quantile, threshold in zip(threshold_quantiles, thresholds):
    for method, distances in {"Exact GP": exact_distances, "DeepRV": deeprv_distances}.items():
        acceptance_probability = np.mean(distances <= threshold, axis=1)
        if acceptance_probability.sum() == 0:
            raise ValueError(f"No {method} simulations passed the {quantile:.0%} threshold")
        method_weights = acceptance_probability / acceptance_probability.sum()
        weights[(quantile, method)] = method_weights
        diagnostic_rows.append({"Threshold Quantile": quantile, "Threshold": threshold, "Method": method, "Overall Acceptance Rate": acceptance_probability.mean(), "Active Parameters": np.count_nonzero(acceptance_probability), "ESS": 1 / np.sum(method_weights ** 2)})

weight_diagnostics = pd.DataFrame(diagnostic_rows)
display(weight_diagnostics.assign(**{"Threshold Quantile": lambda frame: frame["Threshold Quantile"].map(lambda value: f"{value:.0%}"), "Overall Acceptance Rate": lambda frame: frame["Overall Acceptance Rate"].map(lambda value: f"{value:.2%}")}))

## Particle-Level Acceptance Agreement

Each point below is one complete, matched parameter combination. For each Exact-GP distance quantile, its coordinates are the fractions of that particle's 50 simulations that pass the same numerical distance threshold under Exact GP and DeepRV. The dashed line denotes equal acceptance probability. Because there are 50 replicates, both coordinates occur in increments of 0.02.


In [ ]:
acceptance_quantiles = np.array([0.20, 0.10, 0.05, 0.01])

figure, axes = plt.subplots(2, 2, figsize=(5.906, 5.4), sharex=True, sharey=True, constrained_layout=True)
for axis, quantile in zip(axes.ravel(), acceptance_quantiles):
    threshold_index = np.flatnonzero(np.isclose(threshold_quantiles, quantile)).item()
    threshold = thresholds[threshold_index]
    exact_acceptance = np.mean(exact_distances <= threshold, axis=1)
    deeprv_acceptance = np.mean(deeprv_distances <= threshold, axis=1)
    difference = deeprv_acceptance - exact_acceptance
    correlation = np.corrcoef(exact_acceptance, deeprv_acceptance)[0, 1]

    axis.scatter(exact_acceptance, deeprv_acceptance, s=7, color=method_styles["Exact GP"]["color"], alpha=0.2, edgecolors="none", rasterized=True)
    axis.plot([0, 1], [0, 1], color="#4D4D4D", linestyle="--", linewidth=1.0)
    axis.set_title(rf"${quantile:.2f}$ Quantile", fontsize=11)
    axis.set_xlim(-0.02, 1.02)
    axis.set_ylim(-0.02, 1.02)
    axis.set_xticks([0, 0.5, 1])
    axis.set_yticks([0, 0.5, 1])
    axis.set_aspect("equal", adjustable="box")
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.tick_params(labelsize=9, direction="out")

figure.supxlabel("Exact-GP Acceptance Probability", fontsize=11)
figure.supylabel("DeepRV Acceptance Probability", fontsize=11)
figure.savefig(result_directory / "age_size_acceptance_probability_agreement.pdf", bbox_inches="tight")
figure.savefig(result_directory / "age_size_acceptance_probability_agreement.png", dpi=300, bbox_inches="tight")
plt.show()


## Build One Age–Size IPM Per Parameter Combination

This section follows `experiments/soay/age_size/abc/Make_IPM_agesize.ipynb`. It uses a 250-point size mesh, ages 0–7 and the age-8 greybeard class, giving 2,250 states. The DeepRV decoder is not called here: its effect is already represented by the DeepRV rejection weights.

Vital-rate predictions and scalar IPM scores are cached. Only the scores needed for the offline comparison are retained, rather than storing three 2,250-element eigenvectors for every parameter combination.

In [ ]:
data = pd.read_csv(data_file)
with np.load(samples_file) as saved:
    gp_samples = {name: saved[name].copy() for name in saved.files if not name.endswith("_latent")}
with np.load(true_metrics_file) as saved:
    true_metrics = {name: saved[name].copy() for name in saved.files}

survival = data[["z", "age", "Surv"]].dropna()
reproduction = data.loc[data["age"] >= minimum_reproductive_age, ["z", "age", "Repr"]].dropna()
growth = data[["z", "age", "z1"]].dropna()
recruitment = data[["age", "Recr"]].dropna()
recruit_size = data[["z", "Rcsz"]].dropna()
training_inputs = {
    "surv": survival[["z", "age"]].to_numpy(np.float64),
    "repr": reproduction[["z", "age"]].to_numpy(np.float64),
    "growth": growth[["z", "age"]].to_numpy(np.float64),
    "recr": recruitment[["age"]].to_numpy(np.float64),
    "rcsz": recruit_size[["z"]].to_numpy(np.float64),
}
for rate, inputs in training_inputs.items():
    if inputs.shape[0] != gp_samples[f"{rate}_mu"].shape[1]:
        raise ValueError(f"{rate} training inputs do not match the saved GP values")

number_ages = maximum_natural_age + 2
number_states = number_ages * number_mesh_points
ages = np.arange(number_ages)
mesh_width = (upper_size - lower_size) / number_mesh_points
mesh = lower_size + (np.arange(number_mesh_points) + 0.5) * mesh_width
age_size_locations = np.column_stack([np.tile(mesh, number_ages), np.repeat(ages, number_mesh_points)])
prediction_locations = {"growth": age_size_locations, "surv": age_size_locations, "repr": age_size_locations, "recr": ages[:, None], "rcsz": mesh[:, None]}
prediction_shapes = {"growth": (number_ages, number_mesh_points), "surv": (number_ages, number_mesh_points), "repr": (number_ages, number_mesh_points), "recr": (number_ages,), "rcsz": (number_mesh_points,)}
prediction_fields = {"growth": ("mean", "variance"), "surv": ("mean",), "repr": ("mean",), "recr": ("mean",), "rcsz": ("mean", "variance")}
used_indices = {rate: np.unique(parameter_indices[:, column]) for rate, column in rate_columns.items()}

if np.asarray(true_metrics["omega"]).size != number_states:
    raise ValueError(f"True IPM metrics contain {np.asarray(true_metrics['omega']).size} states; expected {number_states}")
print(f"Age Classes: {number_ages} ({ages[0]}–{ages[-1]}, With Age {ages[-1]} As Greybeard)")
print(f"IPM Dimensions: {number_states:,} × {number_states:,}")
display(pd.DataFrame({"Observations": {rate: len(values) for rate, values in training_inputs.items()}, "Input Dimensions": {rate: values.shape[1] for rate, values in training_inputs.items()}, "Unique Posterior Indices": {rate: len(values) for rate, values in used_indices.items()}}))

In [ ]:
class MCMCConditionedGP:
    def __init__(self, inputs, function_values, variance, lengthscales, intercept, likelihood, jitter=5e-4):
        dtype = gpflow.default_float()
        inputs = np.asarray(inputs, dtype=np.float64)
        if inputs.ndim == 1:
            inputs = inputs.reshape(-1, 1)
        self.input_dimension = inputs.shape[1]
        self.inputs = tf.convert_to_tensor(inputs, dtype=dtype)
        self.intercept = tf.convert_to_tensor(intercept, dtype=dtype)
        self.likelihood = likelihood
        lengthscales = np.asarray(lengthscales, dtype=np.float64).reshape(-1)
        if lengthscales.size != self.input_dimension:
            raise ValueError(f"Received {lengthscales.size} lengthscales for {self.input_dimension} inputs")
        centered = np.asarray(function_values, dtype=np.float64).reshape(-1, 1) - float(intercept)
        self.centered = tf.convert_to_tensor(centered, dtype=dtype)
        self.kernel = gpflow.kernels.SquaredExponential(variance=float(variance), lengthscales=lengthscales)
        covariance = self.kernel(self.inputs, self.inputs) + tf.cast(jitter, dtype) * tf.eye(len(inputs), dtype=dtype)
        self.cholesky = tf.linalg.cholesky(covariance)
        self.alpha = tf.linalg.cholesky_solve(self.cholesky, self.centered)

    def prepare_inputs(self, inputs):
        inputs = tf.convert_to_tensor(inputs, dtype=gpflow.default_float())
        if inputs.shape.rank == 0:
            inputs = tf.reshape(inputs, (1, 1))
        elif inputs.shape.rank == 1:
            inputs = tf.reshape(inputs, (-1, 1)) if self.input_dimension == 1 else tf.reshape(inputs, (1, -1))
        if inputs.shape.rank != 2 or inputs.shape[1] != self.input_dimension:
            raise ValueError(f"Expected {self.input_dimension} input columns, received shape {inputs.shape}")
        return inputs

    def predict_f(self, inputs):
        inputs = self.prepare_inputs(inputs)
        cross_covariance = self.kernel(self.inputs, inputs)
        mean = tf.linalg.matmul(cross_covariance, self.alpha, transpose_a=True) + self.intercept
        solved = tf.linalg.triangular_solve(self.cholesky, cross_covariance)
        variance = self.kernel.K_diag(inputs) - tf.reduce_sum(tf.square(solved), axis=0)
        return mean, tf.reshape(tf.maximum(variance, tf.cast(0, variance.dtype)), (-1, 1))

    def predict_y(self, inputs):
        inputs = self.prepare_inputs(inputs)
        mean, variance = self.predict_f(inputs)
        return self.likelihood.predict_mean_and_var(inputs, mean, variance)

def predict_vital_rate(rate, sample_index):
    if rate in {"surv", "repr", "recr"}:
        likelihood = gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid)
    else:
        likelihood = gpflow.likelihoods.Gaussian(variance=float(gp_samples[f"{rate}_noise"][sample_index]) ** 2)
    model = MCMCConditionedGP(training_inputs[rate], gp_samples[f"{rate}_mu"][sample_index], gp_samples[f"{rate}_var"][sample_index], gp_samples[f"{rate}_ls"][sample_index], gp_samples[f"{rate}_int"][sample_index], likelihood)
    mean, variance = model.predict_y(prediction_locations[rate])
    shape = prediction_shapes[rate]
    return mean.numpy().reshape(shape), variance.numpy().reshape(shape)

In [ ]:
vital_rate_predictions = {}
if prediction_cache_file.exists():
    with np.load(prediction_cache_file) as cache:
        cache_matches = str(cache["design_hash"].item()) == design_hash and int(cache["number_mesh_points"].item()) == number_mesh_points and float(cache["lower_size"].item()) == lower_size and float(cache["upper_size"].item()) == upper_size and int(cache["maximum_natural_age"].item()) == maximum_natural_age
        if not cache_matches:
            raise ValueError("The vital-rate cache does not match this validation design or mesh")
        for rate in rate_columns:
            cached_indices = cache[f"{rate}_indices"].astype(int)
            if not np.array_equal(cached_indices, used_indices[rate]):
                raise ValueError(f"The vital-rate cache uses different {rate} indices")
            vital_rate_predictions[rate] = {field: cache[f"{rate}_{field}"].copy() for field in prediction_fields[rate]}
    print(f"Loaded {prediction_cache_file.name}")
else:
    for rate in rate_columns:
        shape = (len(used_indices[rate]),) + prediction_shapes[rate]
        vital_rate_predictions[rate] = {field: np.empty(shape) for field in prediction_fields[rate]}
        for position, sample_index in enumerate(used_indices[rate], start=1):
            mean, variance = predict_vital_rate(rate, sample_index)
            vital_rate_predictions[rate]["mean"][position - 1] = mean
            if "variance" in prediction_fields[rate]:
                vital_rate_predictions[rate]["variance"][position - 1] = variance
            if position % 100 == 0 or position == len(used_indices[rate]):
                print(f"{rate}: {position:,}/{len(used_indices[rate]):,} Predictions", flush=True)
                gc.collect()
    cache_values = {f"{rate}_{field}": values for rate, predictions in vital_rate_predictions.items() for field, values in predictions.items()}
    cache_values.update({f"{rate}_indices": values for rate, values in used_indices.items()})
    np.savez_compressed(prediction_cache_file, **cache_values, design_hash=np.asarray(design_hash), number_mesh_points=np.asarray(number_mesh_points), lower_size=np.asarray(lower_size), upper_size=np.asarray(upper_size), maximum_natural_age=np.asarray(maximum_natural_age))
    print(f"Saved {prediction_cache_file.name}")

In [ ]:
def correct_columns(matrix, target):
    column_sums = matrix.sum(axis=0)
    matrix *= np.divide(target, column_sums, out=np.ones_like(column_sums), where=column_sums > 0)

def make_kernel(predictions):
    kernel = np.zeros((number_states, number_states))
    reproduction_probability = np.zeros(number_states)
    age_zero_rows = slice(0, number_mesh_points)
    recruit_sd = np.sqrt(np.maximum(predictions["rcsz"]["variance"], 1e-12))
    recruit_density = norm.pdf(mesh[:, None], loc=predictions["rcsz"]["mean"][None, :], scale=recruit_sd[None, :])

    for age in ages:
        source_columns = slice(age * number_mesh_points, (age + 1) * number_mesh_points)
        next_age = age + 1 if age < number_ages - 1 else age
        survival_rows = slice(next_age * number_mesh_points, (next_age + 1) * number_mesh_points)
        survival = np.clip(predictions["surv"]["mean"][age], 0, 1)
        growth_sd = np.sqrt(np.maximum(predictions["growth"]["variance"][age], 1e-12))
        growth_density = norm.pdf(mesh[:, None], loc=predictions["growth"]["mean"][age][None, :], scale=growth_sd[None, :])
        survival_block = mesh_width * survival[None, :] * growth_density
        correct_columns(survival_block, survival)
        kernel[survival_rows, source_columns] = survival_block

        if age >= minimum_reproductive_age:
            reproduction = np.clip(predictions["repr"]["mean"][age], 0, 1)
            recruitment = float(np.clip(predictions["recr"]["mean"][age], 0, 1))
            fecundity_target = survival * reproduction * female_probability * recruitment
            fecundity_block = mesh_width * fecundity_target[None, :] * recruit_density
            correct_columns(fecundity_block, fecundity_target)
            kernel[age_zero_rows, source_columns] += fecundity_block
            reproduction_probability[source_columns] = reproduction

    return kernel, reproduction_probability

def calculate_ipm_metrics_sparse(kernel, reproduction_probability, tolerance=1e-9):
    matrix = csr_matrix(kernel)
    initial_vector = np.ones(matrix.shape[0])
    eigenvalues, right_vectors = eigs(matrix, k=1, which="LR", v0=initial_vector, tol=tolerance)
    _, left_vectors = eigs(matrix.T, k=1, which="LR", v0=initial_vector, tol=tolerance)
    lambda_value = float(eigenvalues[0].real)
    stable_state = right_vectors[:, 0].real
    stable_state *= -1 if stable_state.sum() < 0 else 1
    stable_state = np.maximum(stable_state, 0)
    stable_state /= stable_state.sum()
    reproductive_value = left_vectors[:, 0].real
    reproductive_value *= -1 if reproductive_value.sum() < 0 else 1
    if np.isclose(reproductive_value[0], 0):
        raise RuntimeError("The first reproductive-value element is numerically zero")
    reproductive_value /= reproductive_value[0]
    reproductive_stable_state = reproduction_probability * stable_state
    if reproductive_stable_state.sum() <= 0:
        raise RuntimeError("The reproductive stable distribution has zero mass")
    reproductive_stable_state /= reproductive_stable_state.sum()
    return {"lambda": lambda_value, "omega": stable_state, "omega_rep": reproductive_stable_state, "gamma": reproductive_value}

def normalise(values, epsilon=1e-12):
    values = np.clip(np.asarray(values, dtype=float), epsilon, None)
    return values / values.sum()

true_lambda = float(true_metrics["lambda"])
true_stable_state = normalise(true_metrics["omega"])
true_reproductive_stable_state = normalise(true_metrics["omega_rep"])
true_reproductive_value = np.asarray(true_metrics["gamma"], dtype=float)

def score_ipm(metrics):
    stable_state = normalise(metrics["omega"])
    reproductive_stable_state = normalise(metrics["omega_rep"])
    reproductive_value = np.asarray(metrics["gamma"], dtype=float)
    cosine = np.sum(reproductive_value * true_reproductive_value) / (np.linalg.norm(reproductive_value) * np.linalg.norm(true_reproductive_value))
    return {
        "lambda": metrics["lambda"],
        "lambda_mse": (metrics["lambda"] - true_lambda) ** 2,
        "stable_state_kl": np.sum(true_stable_state * np.log(true_stable_state / stable_state)),
        "reproductive_value_cosine": cosine,
        "reproductive_stable_distribution_kl": np.sum(true_reproductive_stable_state * np.log(true_reproductive_stable_state / reproductive_stable_state)),
    }

In [ ]:
score_keys = ["lambda", "lambda_mse", "stable_state_kl", "reproductive_value_cosine", "reproductive_stable_distribution_kl"]
if ipm_cache_file.exists():
    with np.load(ipm_cache_file) as cache:
        cache_matches = str(cache["design_hash"].item()) == design_hash and int(cache["number_mesh_points"].item()) == number_mesh_points and float(cache["lower_size"].item()) == lower_size and float(cache["upper_size"].item()) == upper_size and int(cache["maximum_natural_age"].item()) == maximum_natural_age
        if not cache_matches:
            raise ValueError("The IPM score cache does not match this validation design or mesh")
        ipm_scores = {name: cache[name].copy() for name in score_keys}
    print(f"Loaded {ipm_cache_file.name}")
else:
    ipm_scores = {name: np.empty(number_parameters) for name in score_keys}
    for parameter_id, indices in enumerate(parameter_indices):
        predictions = {}
        for rate, column in rate_columns.items():
            position = np.searchsorted(used_indices[rate], indices[column])
            predictions[rate] = {field: values[position] for field, values in vital_rate_predictions[rate].items()}
        kernel, reproduction_probability = make_kernel(predictions)
        scores = score_ipm(calculate_ipm_metrics_sparse(kernel, reproduction_probability))
        for name in score_keys:
            ipm_scores[name][parameter_id] = scores[name]
        del predictions, kernel, reproduction_probability
        if (parameter_id + 1) % 25 == 0 or parameter_id + 1 == number_parameters:
            print(f"IPMs: {parameter_id + 1:,}/{number_parameters:,}", flush=True)
            gc.collect()
    np.savez_compressed(ipm_cache_file, **ipm_scores, design_hash=np.asarray(design_hash), number_mesh_points=np.asarray(number_mesh_points), lower_size=np.asarray(lower_size), upper_size=np.asarray(upper_size), maximum_natural_age=np.asarray(maximum_natural_age))
    print(f"Saved {ipm_cache_file.name}")

for name, values in ipm_scores.items():
    if values.shape != (number_parameters,) or not np.isfinite(values).all():
        raise ValueError(f"Invalid cached IPM score: {name}")

## Weighted Point Estimates

Each loss is calculated for every IPM before averaging with the rejection weights. Lower is better for the three loss metrics, while higher is better for reproductive-value cosine similarity.

In [ ]:
particle_scores = {
    "Lambda MSE": ipm_scores["lambda_mse"],
    "Stable State KL": ipm_scores["stable_state_kl"],
    "Reproductive Value Cosine Similarity": ipm_scores["reproductive_value_cosine"],
    "Reproductive Stable Distribution KL": ipm_scores["reproductive_stable_distribution_kl"],
}

result_rows = []
for quantile, threshold in zip(threshold_quantiles, thresholds):
    for method in method_styles:
        method_weights = weights[(quantile, method)]
        lambda_mean = np.sum(method_weights * ipm_scores["lambda"])
        row = {"Threshold Quantile": quantile, "Threshold": threshold, "Method": method, "Lambda Mean": lambda_mean, "Lambda Standard Deviation": np.sqrt(np.sum(method_weights * (ipm_scores["lambda"] - lambda_mean) ** 2))}
        row.update({name: np.sum(method_weights * values) for name, values in particle_scores.items()})
        result_rows.append(row)

point_estimates = pd.DataFrame(result_rows).merge(weight_diagnostics, on=["Threshold Quantile", "Threshold", "Method"], validate="one_to_one")
display(point_estimates.round(6))

In [ ]:
comparison_metrics = ["Lambda MSE", "Stable State KL", "Reproductive Value Cosine Similarity", "Reproductive Stable Distribution KL"]
method_difference = point_estimates.pivot(index="Threshold Quantile", columns="Method", values=comparison_metrics)
method_difference = method_difference.xs("DeepRV", axis=1, level=1) - method_difference.xs("Exact GP", axis=1, level=1)
method_difference.index = method_difference.index.map(lambda value: f"{value:.0%}")
method_difference.index.name = "Threshold Quantile"
print("DeepRV Minus Exact GP")
display(method_difference.round(6))

A negative difference favours DeepRV for the three loss metrics. A positive difference favours DeepRV for reproductive-value cosine similarity. These are descriptive point differences, not uncertainty intervals or significance tests.

## Weighted Population Growth-Rate Distributions

Both methods use the same 5,000 population growth-rate values; only their rejection weights differ. Dashed vertical lines show the weighted means.

In [ ]:
def weighted_density(grid, values, weights, bandwidth):
    differences = (grid[:, None] - values[None, :]) / bandwidth
    return norm.pdf(differences) @ weights / bandwidth

def density_grid(values, points=400):
    padding = max(0.03 * np.ptp(values), 1e-8)
    grid = np.linspace(values.min() - padding, values.max() + padding, points)
    bandwidth = max(1.06 * np.std(values, ddof=1) * len(values) ** (-1 / 5), 1e-8)
    return grid, bandwidth

method_styles = {
    "Exact GP": {"color": "#0072B2", "linestyle": "-"},
    "DeepRV": {"color": "#D55E00", "linestyle": "-"},
}

lambda_values = ipm_scores["lambda"]
active = np.zeros(number_parameters, dtype=bool)

for method_weights in weights.values():
    active |= method_weights > 0

lambda_grid, lambda_bandwidth = density_grid(lambda_values[active])

plt.style.use("seaborn-v0_8-white")
figure, axes = plt.subplots(
    2,
    3,
    figsize=(5.906, 5.2),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

for axis, quantile in zip(axes.ravel(), threshold_quantiles):
    for row, (method, style) in enumerate(method_styles.items()):
        method_weights = weights[(quantile, method)]
        density = weighted_density(lambda_grid, lambda_values, method_weights, lambda_bandwidth)
        lambda_mean = np.sum(method_weights * lambda_values)

        axis.fill_between(lambda_grid, density, color=style["color"], alpha=0.10)
        axis.plot(lambda_grid, density, color=style["color"], linestyle=style["linestyle"], linewidth=1.5, label=method)
        axis.axvline(lambda_mean, color=style["color"], linestyle=":", linewidth=1.2)
    axis.set_title(f"{quantile:.0%} Threshold", fontsize=11)
    axis.tick_params(labelsize=9, direction="out")
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

axes.ravel()[-1].remove()

handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="outside upper center", ncol=2, frameon=False, fontsize=9)
figure.supxlabel(r"Population Growth Rate $\lambda$", fontsize=11)
figure.supylabel("Weighted Density", fontsize=11)

figure.savefig(result_directory / "agesize_offline_weighted_lambda_distributions_all_thresholds.pdf", bbox_inches="tight")
figure.savefig(result_directory / "agesize_offline_weighted_lambda_distributions_all_thresholds.png", dpi=300, bbox_inches="tight")

plt.show()

## Weighted IPM-Score Distributions

In [ ]:
metrics_to_plot = ["Stable State KL", "Reproductive Value Cosine Similarity", "Reproductive Stable Distribution KL"]

for metric in metrics_to_plot:
    metric_values = particle_scores[metric]
    active = np.zeros(len(metric_values), dtype=bool)
    for method_weights in weights.values():
        active |= method_weights > 0
    plot_values = metric_values[active]
    metric_grid, bandwidth = density_grid(plot_values)
    if "KL" in metric:
        metric_grid = np.linspace(max(0, metric_grid.min()), metric_grid.max(), len(metric_grid))
    else:
        metric_grid = np.linspace(max(-1, metric_grid.min()), min(1, metric_grid.max()), len(metric_grid))

    figure, axes = plt.subplots(2, 3, figsize=(figure_width, 5.2), sharex=True, sharey=True, constrained_layout=True)
    for axis, quantile in zip(axes.ravel(), threshold_quantiles):
        for row, (method, style) in enumerate(method_styles.items()):
            method_weights = weights[(quantile, method)]
            density = weighted_density(metric_grid, metric_values, method_weights, bandwidth)
            weighted_mean = np.sum(method_weights * metric_values)
            axis.fill_between(metric_grid, density, color=style["color"], alpha=0.14)
            axis.plot(metric_grid, density, color=style["color"], linestyle=style["linestyle"], linewidth=1.5, label=method)
            axis.axvline(weighted_mean, color=style["color"], linestyle=":", linewidth=1.2)
            axis.text(0.03, 0.96 - 0.11 * row, f"{method}: {weighted_mean:.4g}", transform=axis.transAxes, color=style["color"], fontsize=7, va="top")
        axis.set_title(f"{quantile:.0%} Threshold", fontsize=11)
        axis.tick_params(labelsize=9, direction="out")
        axis.grid(False)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)
    axes.ravel()[-1].set_visible(False)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="outside upper center", ncol=2, frameon=False, fontsize=9)
    figure.supxlabel(metric, fontsize=11)
    figure.supylabel("Weighted Density", fontsize=11)
    file_name = metric.lower().replace(" ", "_")
    figure.savefig(result_directory / f"offline_weighted_{file_name}_distributions.pdf", bbox_inches="tight")
    figure.savefig(result_directory / f"offline_weighted_{file_name}_distributions.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
def weighted_density(grid, values, method_weights, bandwidth):
    differences = (grid[:, None] - values[None, :]) / bandwidth
    return norm.pdf(differences) @ method_weights / bandwidth

def density_grid(values, points=400):
    padding = max(0.03 * np.ptp(values), 1e-8)
    grid = np.linspace(values.min() - padding, values.max() + padding, points)
    bandwidth = max(1.06 * np.std(values, ddof=1) * len(values) ** (-1 / 5), 1e-8)
    return grid, bandwidth

method_styles = {
    "Exact GP": {"color": "#0072B2", "linestyle": "-"},
    "DeepRV": {"color": "#D55E00", "linestyle": "-"},
}

metrics_to_plot = {
    "Lambda MSE": ipm_scores["lambda_mse"],
    "Stable State KL": ipm_scores["stable_state_kl"],
    "Reproductive Value\nCosine Similarity": ipm_scores["reproductive_value_cosine"],
    "Reproductive Stable\nDistribution KL": ipm_scores["reproductive_stable_distribution_kl"],
}

selected_quantile = 0.01
active = (weights[(selected_quantile, "Exact GP")] > 0) | (weights[(selected_quantile, "DeepRV")] > 0)

plt.style.use("seaborn-v0_8-white")
figure, axes = plt.subplots(2, 2, figsize=(5.906, 4.8), constrained_layout=True)

for axis, (metric, metric_values) in zip(axes.ravel(), metrics_to_plot.items()):
    metric_grid, bandwidth = density_grid(metric_values[active])

    if "KL" in metric or "MSE" in metric:
        metric_grid = np.linspace(max(0, metric_grid.min()), metric_grid.max(), len(metric_grid))
    else:
        metric_grid = np.linspace(max(-1, metric_grid.min()), min(1, metric_grid.max()), len(metric_grid))

    for method, style in method_styles.items():
        method_weights = weights[(selected_quantile, method)]
        density = weighted_density(metric_grid, metric_values, method_weights, bandwidth)
        weighted_mean = np.sum(method_weights * metric_values)

        axis.fill_between(metric_grid, density, color=style["color"], alpha=0.10)
        axis.plot(metric_grid, density, color=style["color"], linestyle=style["linestyle"], linewidth=1.5, label=method)
        axis.axvline(weighted_mean, color=style["color"], linestyle=":", linewidth=1.2)

    axis.set_xlabel(metric, fontsize=9)
    axis.tick_params(labelsize=9, direction="out")
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="outside upper center", ncol=2, frameon=False, fontsize=9)
figure.supylabel("Weighted Density", fontsize=11)

figure.savefig(result_directory / "agesize_offline_weighted_ipm_metrics_1_percent.pdf", bbox_inches="tight")
figure.savefig(result_directory / "agesize_offline_weighted_ipm_metrics_1_percent.png", dpi=300, bbox_inches="tight")

plt.show()

## Save Results

The point estimates, diagnostics, method differences and normalized weights are saved so later analysis does not need to reopen the raw chunks or rebuild the IPMs.

In [ ]:
point_estimates.to_csv(point_estimate_file, index=False)
weight_diagnostics.to_csv(diagnostic_file, index=False)
method_difference.to_csv(difference_file)
weight_values = {f"{method.lower().replace(' ', '_')}_{int(100 * quantile):02d}": values for (quantile, method), values in weights.items()}
np.savez_compressed(weight_file, **weight_values, parameter_indices=parameter_indices, threshold_quantiles=threshold_quantiles, thresholds=thresholds, design_hash=np.asarray(design_hash), informative_components=informative_components)
print(f"Saved {point_estimate_file.name}")
print(f"Saved {diagnostic_file.name}")
print(f"Saved {difference_file.name}")
print(f"Saved {weight_file.name}")